# GSHHG Semantics

## Objective

Understand the semantic structure of GSHHG before performing geometry analysis.

We want to determine:

1. What do the resolution levels `c`, `l`, `i`, `h`, `f` mean?
2. What do shoreline levels `L1-L6` represent?
3. What attributes are available in the shapefile?
4. What does each attribute mean?
5. How are polygons related hierarchically?
6. Which semantic information is relevant to web tiles?
7. Which information should be preserved or discarded during preprocessing?

In [1]:
from pathlib import Path
from collections import Counter
import re

import pandas as pd

DATA_DIR = Path(r"P:\HACKATHONS\SIH26PS67\DATA\REF")
COASTLINE_DIR = DATA_DIR / "coastline"
GSHHG_DIR = COASTLINE_DIR / "gshhg"   # notice the G at the end

GSHHS_DIR = GSHHG_DIR / "GSHHS_shp"   # notice the S at the end
WDBII_DIR = GSHHG_DIR / "WDBII_shp"

In [2]:
RESOLUTION_INFO = {
    "f": {
        "name": "Full",
        "description": "Maximum resolution; original data has not been decimated."
    },
    "h": {
        "name": "High",
        "description": "Reduced from full resolution using Douglas-Peucker simplification."
    },
    "i": {
        "name": "Intermediate",
        "description": "Reduced from high resolution using Douglas-Peucker simplification."
    },
    "l": {
        "name": "Low",
        "description": "Reduced from intermediate resolution using Douglas-Peucker simplification."
    },
    "c": {
        "name": "Crude",
        "description": "Reduced from low resolution using Douglas-Peucker simplification."
    },
}

LEVEL_INFO = {
    1: {
        "name": "Land",
        "description": "Continental land masses and ocean islands, except Antarctica."
    },
    2: {
        "name": "Lake",
        "description": "Lakes."
    },
    3: {
        "name": "Island in lake",
        "description": "Islands contained within lakes."
    },
    4: {
        "name": "Pond in island in lake",
        "description": "Ponds contained within islands inside lakes."
    },
    5: {
        "name": "Antarctica Ice",
        "description": "Boundaries of Antarctica Ice and Ocean."
    },
    6: {
        "name": "Antarctica Grounds",
        "description": "Boundaries of Antarctica Grounds and Ocean."
    }
}

In [3]:
resolution_semantics = pd.DataFrame([
    {
        "code": code,
        "name": info["name"],
        "description": info["description"]
    }
    for code, info in RESOLUTION_INFO.items()
])

resolution_semantics

,code,name,description
0,f,Full,Maximum resolution; original data has not been...
1,h,High,Reduced from full resolution using Douglas-Peu...
2,i,Intermediate,Reduced from high resolution using Douglas-Peu...
3,l,Low,Reduced from intermediate resolution using Dou...
4,c,Crude,Reduced from low resolution using Douglas-Peuc...


In [4]:
level_semantics = pd.DataFrame([
    {
        "level": level,
        "name": info["name"],
        "description": info["description"]
    }
    for level, info in LEVEL_INFO.items()
])

level_semantics

,level,name,description
0,1,Land,"Continental land masses and ocean islands, exc..."
1,2,Lake,Lakes.
2,3,Island in lake,Islands contained within lakes.
3,4,Pond in island in lake,Ponds contained within islands inside lakes.
4,5,Antarctica Ice,Boundaries of Antarctica Ice and Ocean.
5,6,Antarctica Grounds,Boundaries of Antarctica Grounds and Ocean.


In [5]:
import geopandas as gpd

In [6]:
REPRESENTATIVE_RESOLUTION = "c"
REPRESENTATIVE_LEVEL = 1

representative_path = (
    GSHHS_DIR
    / REPRESENTATIVE_RESOLUTION
    / f"GSHHS_{REPRESENTATIVE_RESOLUTION}_L{REPRESENTATIVE_LEVEL}.shp"
)

print(representative_path)

P:\HACKATHONS\SIH26PS67\DATA\REF\coastline\gshhg\GSHHS_shp\c\GSHHS_c_L1.shp


In [7]:
print("Exists:", representative_path.exists())
print("Size:", representative_path.stat().st_size / (1024 ** 2), "MB")

Exists: True
Size: 0.15083694458007812 MB


In [8]:
gdf = gpd.read_file(representative_path)

print(f"Features: {len(gdf):,}")

Features: 742


In [9]:
gdf.columns.tolist()

['id', 'level', 'source', 'parent_id', 'sibling_id', 'area', 'geometry']

In [10]:
schema = pd.DataFrame({
    "column": gdf.columns,
    "dtype": [
        str(gdf[column].dtype)
        for column in gdf.columns
    ]
})

schema

,column,dtype
0,id,str
1,level,int32
2,source,str
3,parent_id,int32
4,sibling_id,int32
5,area,float64
6,geometry,geometry


In [11]:
print("Geometry column:", gdf.geometry.name)
print("Geometry type:", gdf.geometry.geom_type.unique())

Geometry column: geometry
Geometry type: <StringArray>
['Polygon']
Length: 1, dtype: str


In [12]:
gdf.geometry.geom_type.value_counts()

Polygon    742
Name: count, dtype: int64

In [13]:
print(gdf.crs)

EPSG:4326


In [14]:
gdf.head()

,id,level,source,parent_id,sibling_id,area,geometry
0,0-E,1,WVS,-1,0,5.065405e+07,"POLYGON ((180 68.99378, 180 65.08455, 179.5250..."
1,0-W,1,WVS,-1,0,5.065405e+07,"POLYGON ((-180 68.99378, -179.99844 68.99292, ..."
2,1,1,WVS,-1,1,2.922097e+07,"POLYGON ((31.91917 31.53122, 32.2845 31.23281,..."
3,2,1,WVS,-1,2,2.015474e+07,"POLYGON ((-81.08258 8.80839, -79.9095 9.35944,..."
4,3,1,WVS,-1,3,1.753416e+07,"POLYGON ((-73.36172 -53.00042, -73.36261 -53, ..."


In [15]:
for column in gdf.columns:
    if column == gdf.geometry.name:
        continue

    nunique = gdf[column].nunique(dropna=False)

    print(f"{column:20} unique values: {nunique:,}")

id                   unique values: 742
level                unique values: 1
source               unique values: 2
parent_id            unique values: 1
sibling_id           unique values: 738
area                 unique values: 738


In [16]:
for column in gdf.columns:
    if column == gdf.geometry.name:
        continue

    nunique = gdf[column].nunique(dropna=False)

    if nunique <= 20:
        print(gdf[column].value_counts(dropna=False))
        print()

level
1    742
Name: count, dtype: int64

source
WVS      740
WDBII      2
Name: count, dtype: int64

parent_id
-1    742
Name: count, dtype: int64



In [17]:
numeric_columns = gdf.select_dtypes(
    include="number"
).columns

numeric_columns.tolist()

['level', 'parent_id', 'sibling_id', 'area']

In [18]:
gdf[numeric_columns].describe().T

,count,mean,std,min,25%,50%,75%,max
level,742.0,1.000000,0.000000e+00,1.000000,1.000000,1.000000,1.000000,1.000000e+00
parent_id,742.0,-1.000000,0.000000e+00,-1.000000,-1.000000,-1.000000,-1.000000,-1.000000e+00
sibling_id,742.0,541.459569,5.482826e+02,0.000000,197.250000,431.500000,714.750000,5.265000e+03
area,742.0,249789.287321,3.010848e+06,11.949314,321.860171,641.613248,2168.766907,5.065405e+07


In [19]:
for column in gdf.columns:
    if column == gdf.geometry.name:
        continue

    unique_count = gdf[column].nunique(dropna=False)

    if unique_count == len(gdf):
        print(f"Potential unique identifier: {column}")

Potential unique identifier: id


In [20]:
if "id" in gdf.columns:
    print(gdf["id"].head(20))
    print("Unique:", gdf["id"].is_unique)

0     0-E
1     0-W
2       1
3       2
4       3
5       6
6       7
7       8
8       9
9      10
10     11
11     12
12     13
13     14
14     15
15     16
16     17
17     18
18     19
19     20
Name: id, dtype: str
Unique: True


In [21]:
hierarchy_candidates = [
    "container",
    "ancestor",
    "level",
    "id"
]

for column in hierarchy_candidates:
    if column in gdf.columns:
        print(f"\n===== {column} =====")
        print(gdf[column].describe())
        print("Unique:", gdf[column].nunique())
        print("Nulls:", gdf[column].isna().sum())


===== level =====
count    742.0
mean       1.0
std        0.0
min        1.0
25%        1.0
50%        1.0
75%        1.0
max        1.0
Name: level, dtype: float64
Unique: 1
Nulls: 0

===== id =====
count     742
unique    742
top       0-E
freq        1
Name: id, dtype: object
Unique: 742
Nulls: 0


In [22]:
area_candidates = [
    column
    for column in gdf.columns
    if "area" in column.lower()
]

area_candidates

['area']

In [23]:
for column in area_candidates:
    print(f"\n===== {column} =====")
    print(gdf[column].describe())


===== area =====
count    7.420000e+02
mean     2.497893e+05
std      3.010848e+06
min      1.194931e+01
25%      3.218602e+02
50%      6.416132e+02
75%      2.168767e+03
max      5.065405e+07
Name: area, dtype: float64


In [24]:
source_candidates = [
    column
    for column in gdf.columns
    if "source" in column.lower()
]

source_candidates

['source']

In [25]:
for column in source_candidates:
    print(f"\n===== {column} =====")
    print(gdf[column].value_counts(dropna=False))


===== source =====
source
WVS      740
WDBII      2
Name: count, dtype: int64


In [26]:
special_candidates = [
    column
    for column in gdf.columns
    if any(
        keyword in column.lower()
        for keyword in ["green", "dateline", "date", "cross"]
    )
]

special_candidates

[]

In [27]:
semantic_inventory = pd.DataFrame({
    "column": gdf.columns,
    "dtype": [
        str(gdf[column].dtype)
        for column in gdf.columns
    ],
    "unique_values": [
        gdf[column].nunique(dropna=False)
        for column in gdf.columns
    ],
    "null_values": [
        gdf[column].isna().sum()
        for column in gdf.columns
    ],
})

semantic_inventory

,column,dtype,unique_values,null_values
0,id,str,742,0
1,level,int32,1,0
2,source,str,2,0
3,parent_id,int32,1,0
4,sibling_id,int32,738,0
5,area,float64,738,0
6,geometry,geometry,742,0


In [28]:
constant_columns = []

for column in gdf.columns:
    if column == gdf.geometry.name:
        continue

    if gdf[column].nunique(dropna=False) == 1:
        constant_columns.append(column)

constant_columns

['level', 'parent_id']

## Web Tile Attribute Strategy

For each source attribute, classify it as:

- REQUIRED
- USEFUL
- OPTIONAL
- REMOVE

Questions:

1. Is this needed to render the coastline?
2. Is this needed for feature identification?
3. Is this needed for interaction/click inspection?
4. Is this needed for filtering?
5. Is this needed for scientific metadata?
6. Can it be inferred from the layer/dataset?
7. Does storing it significantly increase tile size?

In [29]:
attribute_strategy = pd.DataFrame({
    "attribute": gdf.columns,
    "tile_relevance": "UNDECIDED"
})

attribute_strategy

,attribute,tile_relevance
0,id,UNDECIDED
1,level,UNDECIDED
2,source,UNDECIDED
3,parent_id,UNDECIDED
4,sibling_id,UNDECIDED
5,area,UNDECIDED
6,geometry,UNDECIDED


In [30]:
attribute_strategy.loc[
    attribute_strategy["attribute"].str.lower() == "id",
    "tile_relevance"
] = "REQUIRED"

In [31]:
def parse_gshhs_filename(filename):
    match = re.match(
        r"GSHHS_([clihf])_L([1-6])\.shp$",
        filename,
        re.IGNORECASE
    )

    if not match:
        return None

    resolution = match.group(1).lower()
    level = int(match.group(2))

    return {
        "resolution_code": resolution,
        "resolution_name": RESOLUTION_INFO.get(
            resolution, {}
        ).get("name"),
        "level": level,
        "level_name": LEVEL_INFO.get(
            level, {}
        ).get("name", "Other / Antarctica")
    }

In [32]:
parse_gshhs_filename("GSHHS_i_L1.shp")

{'resolution_code': 'i',
 'resolution_name': 'Intermediate',
 'level': 1,
 'level_name': 'Land'}

In [33]:
gshhs_files = sorted(
    path for path in GSHHS_DIR.rglob("*")
    if path.is_file()
)

shp_files = sorted(
    path for path in gshhs_files
    if path.suffix.lower() == ".shp"
)

pattern = re.compile(
    r"GSHHS_([clihf])_L([1-6])\.shp$",
    re.IGNORECASE
)

records = []

for path in shp_files:
    match = pattern.search(path.name)

    if match:
        resolution = match.group(1).lower()
        level = int(match.group(2))

        records.append({
            "resolution": resolution,
            "level": level,
            "filename": path.name,
            "path": path,
        })

dataset_inventory = pd.DataFrame(records)

semantic_datasets = dataset_inventory.copy()

parsed = semantic_datasets["filename"].apply(
    parse_gshhs_filename
)

parsed_df = pd.DataFrame(parsed.tolist())

semantic_datasets = pd.concat(
    [
        semantic_datasets.reset_index(drop=True),
        parsed_df
    ],
    axis=1
)

semantic_datasets

,resolution,level,filename,path,resolution_code,resolution_name,level,level_name
0,c,1,GSHHS_c_L1.shp,P:\HACKATHONS\SIH26PS67\DATA\REF\coastline\gsh...,c,Crude,1,Land
1,c,2,GSHHS_c_L2.shp,P:\HACKATHONS\SIH26PS67\DATA\REF\coastline\gsh...,c,Crude,2,Lake
2,c,3,GSHHS_c_L3.shp,P:\HACKATHONS\SIH26PS67\DATA\REF\coastline\gsh...,c,Crude,3,Island in lake
3,c,5,GSHHS_c_L5.shp,P:\HACKATHONS\SIH26PS67\DATA\REF\coastline\gsh...,c,Crude,5,Antarctica Ice
4,c,6,GSHHS_c_L6.shp,P:\HACKATHONS\SIH26PS67\DATA\REF\coastline\gsh...,c,Crude,6,Antarctica Grounds
5,f,1,GSHHS_f_L1.shp,P:\HACKATHONS\SIH26PS67\DATA\REF\coastline\gsh...,f,Full,1,Land
6,f,2,GSHHS_f_L2.shp,P:\HACKATHONS\SIH26PS67\DATA\REF\coastline\gsh...,f,Full,2,Lake
7,f,3,GSHHS_f_L3.shp,P:\HACKATHONS\SIH26PS67\DATA\REF\coastline\gsh...,f,Full,3,Island in lake
8,f,4,GSHHS_f_L4.shp,P:\HACKATHONS\SIH26PS67\DATA\REF\coastline\gsh...,f,Full,4,Pond in island in lake
9,f,5,GSHHS_f_L5.shp,P:\HACKATHONS\SIH26PS67\DATA\REF\coastline\gsh...,f,Full,5,Antarctica Ice


In [34]:
semantic_matrix = (
    semantic_datasets
    .pivot(
        index="resolution_name",
        columns="level_name",
        values="filename"
    )
)

semantic_matrix

level_name,Antarctica Grounds,Antarctica Ice,Island in lake,Lake,Land,Pond in island in lake
resolution_name,,,,,,
Crude,GSHHS_c_L6.shp,GSHHS_c_L5.shp,GSHHS_c_L3.shp,GSHHS_c_L2.shp,GSHHS_c_L1.shp,NaN
Full,GSHHS_f_L6.shp,GSHHS_f_L5.shp,GSHHS_f_L3.shp,GSHHS_f_L2.shp,GSHHS_f_L1.shp,GSHHS_f_L4.shp
High,GSHHS_h_L6.shp,GSHHS_h_L5.shp,GSHHS_h_L3.shp,GSHHS_h_L2.shp,GSHHS_h_L1.shp,GSHHS_h_L4.shp
Intermediate,GSHHS_i_L6.shp,GSHHS_i_L5.shp,GSHHS_i_L3.shp,GSHHS_i_L2.shp,GSHHS_i_L1.shp,GSHHS_i_L4.shp
Low,GSHHS_l_L6.shp,GSHHS_l_L5.shp,GSHHS_l_L3.shp,GSHHS_l_L2.shp,GSHHS_l_L1.shp,GSHHS_l_L4.shp
